In [1]:
from pathlib import Path
import pandas as pd
import os

BASE_DIR = Path.cwd().parent
KOL_PATH = BASE_DIR / "data" / "kol" / "KOL_Posts.csv"
CRM_PATH = BASE_DIR / "data" / "marketing" / "CRM Data.xlsx"
FB_PATH = BASE_DIR / "data" / "marketing" / "fb_ads_valid.csv"
OUT_DIR = BASE_DIR / "_3_marketing"

ADD KOL TO MASTER

In [2]:
from __future__ import annotations

from pathlib import Path
import hashlib
import numpy as np
import pandas as pd

# -----------------------------
# Helpers
# -----------------------------
def to_num(series: pd.Series) -> pd.Series:
    """Convert strings like '111,112', '0.6%', '  12 ' to numeric floats.
    Percent strings become fractions (0.6% -> 0.006).
    """
    s = series.astype(str).str.strip()
    is_pct = s.str.endswith("%", na=False)
    s = s.str.replace("%", "", regex=False)
    s = s.str.replace(",", "", regex=False)

    out = pd.to_numeric(s, errors="coerce")
    out.loc[is_pct] = out.loc[is_pct] / 100.0
    return out

def parse_date(series: pd.Series) -> pd.Series:
    return pd.to_datetime(series, errors="coerce", dayfirst=True)

def pick_metric(df: pd.DataFrame, candidates: list[str]) -> pd.Series:
    """Return first existing column converted to numeric."""
    for col in candidates:
        if col in df.columns:
            return to_num(df[col])
    return pd.Series([np.nan] * len(df), index=df.index)

def make_activity_id_kol(out: pd.DataFrame, source_name: str = "KOL") -> pd.Series:
    """Stable-ish id from key fields (uses your kol_* column names)."""
    base = (
        out["kol_restaurant_id"].fillna("").astype(str)
        + "|"
        + out["kol_id"].fillna("").astype(str)
        + "|"
        + out["kol_platform"].fillna("").astype(str)
        + "|"
        + out["kol_post_url"].fillna("").astype(str)
        + "|"
        + out["kol_post_date"].dt.strftime("%Y-%m-%d").fillna("")
    )
    return base.apply(lambda s: f"{source_name}_" + hashlib.md5(s.encode("utf-8")).hexdigest()[:12])

# -----------------------------
# Load
# -----------------------------
df = pd.read_csv(KOL_PATH).drop(columns=["Unnamed: 0"], errors="ignore").copy()

# -----------------------------
# Build MASTER (KOL slice)
# -----------------------------
out = pd.DataFrame(index=df.index)

# minimal “master” identifiers
out["source_table"] = "KOL_Posts.csv"
out["channel"] = "KOL"

# KOL core
out["kol_platform"] = df.get("Social Media", np.nan)
out["kol_id"] = df.get("KOL_ID", np.nan)
out["kol_username"] = df.get("Username", np.nan)

# Restaurant keys (still KOL-specific in your design)
out["kol_restaurant_id"] = df.get("Restaurant Code", np.nan)
out["kol_restaurant_name"] = df.get("Restaurant Name", np.nan)
out["kol_country"] = df.get("Country", np.nan)
out["kol_cuisine_type"] = df.get("Cuisine Type", np.nan)

# Post metadata
out["kol_post_url"] = df.get("Post_URL", df.get("Social media URL", np.nan))

# Date: your earlier table didn’t have Post Date; fallback to Booking Date if needed
if "Post Date" in df.columns:
    out["kol_post_date"] = parse_date(df["Post Date"])
else:
    out["kol_post_date"] = parse_date(df.get("Booking Date", pd.Series([np.nan] * len(df), index=df.index)))

# Spend
out["kol_price_per_post"] = to_num(df.get("Price per post", pd.Series([np.nan] * len(df), index=df.index)))
out["kol_cpm"] = to_num(df.get("CPM", pd.Series([np.nan] * len(df), index=df.index)))

# -----------------------------
# KOL metrics (window fallback)
# -----------------------------
out["kol_views"] = pick_metric(df, ["views (latest total)", "Views (+10)", "Views (+5)", "Views"])
out["kol_likes"] = pick_metric(df, ["Likes (+10)", "Likes (+5)", "Likes"])
out["kol_comments"] = pick_metric(df, ["Comments (+10)", "Comments (+5)", "Comments"])

# optional: treat -1 as missing (common encoding)
out.loc[out["kol_likes"] == -1, "kol_likes"] = np.nan
out.loc[out["kol_comments"] == -1, "kol_comments"] = np.nan

out["kol_conv_rate"] = to_num(df.get("Conv Rate", pd.Series([np.nan] * len(df), index=df.index)))
out["kol_growth_rate"] = to_num(df.get("growth rate", pd.Series([np.nan] * len(df), index=df.index)))

# -----------------------------
# Unique activity id
# -----------------------------
out["activity_id"] = make_activity_id_kol(out, source_name="KOL")

# -----------------------------
# Final columns (only what exists)
# -----------------------------
master_cols = [
    "activity_id", "source_table", "channel",
    "kol_platform",
    "kol_id", "kol_username",
    "kol_restaurant_id", "kol_restaurant_name", "kol_country", "kol_cuisine_type",
    "kol_post_url", "kol_post_date",
    "kol_price_per_post", "kol_cpm",
    "kol_views", "kol_likes", "kol_comments",
    "kol_conv_rate", "kol_growth_rate",
]
master_kol = out[master_cols].copy()

print("MASTER_KOL shape:", master_kol.shape)
master_kol


MASTER_KOL shape: (923, 19)


,activity_id,source_table,channel,kol_platform,kol_id,kol_username,kol_restaurant_id,kol_restaurant_name,kol_country,kol_cuisine_type,kol_post_url,kol_post_date,kol_price_per_post,kol_cpm,kol_views,kol_likes,kol_comments,kol_conv_rate,kol_growth_rate
0,KOL_78dd89a55819,KOL_Posts.csv,KOL,Instagram,HH-IN36Y,365days2play,1893,Sirimahannop Asiatique The Riverfront,Bangkok,Thai-European,Instagram.com/365days2play,2024-08-23,NaN,0.0,1723.0,NaN,18.0,0.0099,0.006
1,KOL_d342d0c786bc,KOL_Posts.csv,KOL,Instagram,HH-INAAD,aayan_world,4503,Copper Beyond Buffet Gaysorn Amarin (Hungry Hub),Bangkok,International,Instagram.com/aayan_world,2025-09-05,0.0,0.0,30705.0,NaN,47.0,0.0015,109.054
2,KOL_3a536753bc0b,KOL_Posts.csv,KOL,Instagram,HH-INBEH,beatricechongyh,NaN,Copper Beyond Buffet Gaysorn Amarin,Bangkok,NaN,Instagram.com/beatricechongyh,2025-08-18,NaN,0.0,111112.0,3572.0,75.0,0.0328,0.584
3,KOL_efe32ed42d46,KOL_Posts.csv,KOL,Instagram,HH-INBEH,beatricechongyh,4039,Oishi Grand Siam Paragon,Bangkok,Japanese,Instagram.com/beatricechongyh,2025-08-20,NaN,0.0,102159.0,NaN,70.0,0.0007,0.122
4,KOL_46e9882811d4,KOL_Posts.csv,KOL,Instagram,HH-INBEH,beatricechongyh,5945,Kohaku Omakase,Bangkok,Japanese,Instagram.com/beatricechongyh,2025-08-23,NaN,0.0,102775.0,NaN,44.0,0.0004,0.416
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
918,KOL_d16080ee1c92,KOL_Posts.csv,KOL,Tiktok,NaN,kivalive,NaN,Jim Thompson House,Bangkok,NaN,https://tiktok.com/@kivalive,2025-09-16,NaN,NaN,809.0,NaN,NaN,NaN,-1.000
919,KOL_ff9d0229ca79,KOL_Posts.csv,KOL,NaN,NaN,hunsa.parzing,NaN,Rusty.Bkk,Bangkok,NaN,NaN,2025-11-02,NaN,NaN,NaN,49.0,11.0,NaN,NaN
920,KOL_59d7d64dbcc2,KOL_Posts.csv,KOL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,7.0,0.0,NaN,NaN
921,KOL_59d7d64dbcc2,KOL_Posts.csv,KOL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,3.0,0.0,NaN,NaN


1. BREAKDOWN CRM CAMPAIGN NAME
2. ADD CRM TO MASTER

In [3]:
import pandas as pd
import numpy as np
import hashlib

# -------------------------
# helpers (assumes you already have these; included safe fallbacks)
# -------------------------
def parse_date(s: pd.Series) -> pd.Series:
    """Parse dates/datetimes robustly; returns datetime64[ns] with NaT on errors."""
    return pd.to_datetime(s, errors="coerce")

def to_num(s) -> pd.Series:
    """Coerce to numeric; handles commas/spaces; returns float with NaN on errors."""
    if isinstance(s, pd.Series):
        x = s
    else:
        x = pd.Series(s)
    x = x.astype(str).str.replace(",", "", regex=False).str.strip()
    x = x.replace({"": np.nan, "nan": np.nan, "None": np.nan})
    return pd.to_numeric(x, errors="coerce")

# -------------------------
# parse campaign name (your logic)
# -------------------------
def parse_campaign_name(name):
    if pd.isna(name):
        return pd.Series([np.nan] * 11)

    parts = str(name).split("_")

    # Handle non-standard campaigns (e.g. HNY2025)
    if len(parts) < 10:
        return pd.Series(
            [np.nan, np.nan, np.nan, np.nan, np.nan,
             np.nan, np.nan, np.nan, np.nan, np.nan,
             name]
        )

    return pd.Series([
        parts[0],             # lang
        parts[1],             # city
        parts[2],             # channel
        parts[3],             # platform
        parts[4],             # audience
        parts[5],             # flag1
        parts[6],             # flag2
        parts[7],             # status
        parts[8],             # date
        parts[9],             # time
        "_".join(parts[10:])  # topic
    ])

# -------------------------
# activity id (UPDATED to include restaurant_id)
# -------------------------
def make_activity_id_crm(out: pd.DataFrame, source_name: str = "CRM") -> pd.Series:
    base = (
        out["crm_campaign_id"].fillna("").astype(str)
        + "|"
        + out["crm_sent_datetime"].dt.strftime("%Y-%m-%d %H:%M").fillna("")
        + "|"
        + out["crm_country"].fillna("").astype(str)
        + "|"
        + out["crm_restaurant_id"].fillna("").astype(str)
    )
    return base.apply(lambda s: f"{source_name}_" + hashlib.md5(s.encode("utf-8")).hexdigest()[:12])

# -------------------------
# build master (Option A: explode)
# -------------------------
def build_master_crm(crm_df: pd.DataFrame) -> pd.DataFrame:
    df = crm_df.copy()

    # Optional: normalize column headers (prevents 'Country ' vs 'Country', etc.)
    df.columns = df.columns.astype(str).str.replace("\ufeff", "", regex=False).str.strip()

    # Filter first (so indexes line up)
    df = df.loc[df["Status"].eq("Sent")].copy()

    # Decode campaign name fields (if not already done upstream)
    decoded_cols = [
        "campaign_name_lang",
        "campaign_name_city",
        "campaign_name_channel",
        "campaign_name_platform",
        "campaign_name_audience",
        "campaign_name_flag1",
        "campaign_name_flag2",
        "campaign_name_status",
        "campaign_name_date",
        "campaign_name_time",
        "campaign_name_topic",
    ]
    if "Campaign Name" in df.columns and not all(c in df.columns for c in decoded_cols):
        df[decoded_cols] = df["Campaign Name"].apply(parse_campaign_name)

    out = pd.DataFrame(index=df.index)

    # --- Core ---
    out["source_table"] = "CRM_Data.xlsx"
    out["channel"] = "CRM"

    out["crm_campaign_id"] = df.get("Campaign Id", np.nan)
    out["crm_campaign_name"] = df.get("Campaign Name", np.nan)

    out["crm_country"] = df.get("Country", np.nan)

    # this can be multi-valued like "2449, 2452, 2454"
    out["crm_restaurant_id"] = df.get("rest_id", np.nan)

    # Sent datetime
    out["crm_sent_datetime"] = parse_date(df.get("Sent Date", pd.Series(np.nan, index=df.index)))

    # Campaign metadata (decoded fields)
    out["crm_lang"] = df.get("campaign_name_lang", np.nan)
    out["crm_city"] = df.get("campaign_name_city", np.nan)
    out["crm_audience"] = df.get("campaign_name_audience", np.nan)
    out["crm_topic"] = df.get("campaign_name_topic", np.nan)

    # --- Metrics ---
    out["crm_published"] = to_num(df.get("Published", np.nan))
    out["crm_sent"] = to_num(df.get("Sent", np.nan))
    out["crm_delivered"] = to_num(df.get("Delivered", np.nan))
    out["crm_total_clicked"] = to_num(df.get("Total Clicked", np.nan))
    out["crm_unique_clicked"] = to_num(df.get("Unique Clicked", np.nan))
    out["crm_unique_conversions"] = to_num(df.get("Unique Conversions", np.nan))
    out["crm_revenue"] = to_num(df.get("Revenue", np.nan))

    out["crm_ctr"] = to_num(df.get("CTR", np.nan))
    out["crm_click_to_purchase"] = to_num(df.get("Click_to_purchase", np.nan))
    out["crm_delivered_to_purchase"] = to_num(df.get("delivered_to_purchase", np.nan))
    out["crm_rev_per_click"] = to_num(df.get("Rev_per_click", np.nan))

    # -------------------------
    # Option A: explode restaurants to 1 row per restaurant
    # -------------------------
    out["crm_restaurant_id_raw"] = out["crm_restaurant_id"]

    # Convert to list-of-ids
    # - keep NaN as NaN (don’t turn into "nan" string)
    rid = out["crm_restaurant_id"]
    rid = rid.where(~rid.isna(), np.nan)

    out["crm_restaurant_id"] = (
        rid.astype(str)
           .replace({"nan": np.nan, "None": np.nan})
           .str.split(r"\s*,\s*")  # split commas, trim spaces
    )

    out = out.explode("crm_restaurant_id")

    # Clean numeric
    out["crm_restaurant_id"] = to_num(out["crm_restaurant_id"])

    out["crm_is_restaurant_targeted"] = out["crm_restaurant_id"].notna()
    # -------------------------
    # Unique ID (AFTER explode)
    # -------------------------
    out["activity_id"] = make_activity_id_crm(out)

    master_cols = [
        "activity_id", "source_table", "channel",
        "crm_campaign_id", "crm_campaign_name",
        "crm_country", "crm_restaurant_id", "crm_restaurant_id_raw",
        "crm_sent_datetime",
        "crm_lang", "crm_city",
        "crm_audience", "crm_topic",
        "crm_published", "crm_sent", "crm_delivered",
        "crm_total_clicked", "crm_unique_clicked",
        "crm_unique_conversions", "crm_revenue",
        "crm_ctr", "crm_click_to_purchase",
        "crm_delivered_to_purchase", "crm_rev_per_click",
    ]

    return out[[c for c in master_cols if c in out.columns]].copy()

# -------------------------
# usage
# -------------------------
crm_df = pd.read_excel(CRM_PATH)

master_crm = build_master_crm(crm_df)

print("MASTER_CRM shape:", master_crm.shape)
print(master_crm.head(10))
master_crm


MASTER_CRM shape: (2623, 24)
        activity_id   source_table channel  crm_campaign_id  \
0  CRM_8394b612f8e0  CRM_Data.xlsx     CRM           1350.0   
1  CRM_652ec8390e85  CRM_Data.xlsx     CRM           1351.0   
2  CRM_a09094249fb8  CRM_Data.xlsx     CRM           1353.0   
3  CRM_744905e26ba8  CRM_Data.xlsx     CRM           1352.0   
4  CRM_8f8caca697a6  CRM_Data.xlsx     CRM           1355.0   
5  CRM_c79c1bd0c876  CRM_Data.xlsx     CRM           1354.0   
6  CRM_6b84c52378ad  CRM_Data.xlsx     CRM           1356.0   
7  CRM_1db2908b30ce  CRM_Data.xlsx     CRM           1357.0   
8  CRM_044d68c9632e  CRM_Data.xlsx     CRM           1358.0   
9  CRM_bfe39c999deb  CRM_Data.xlsx     CRM           1359.0   

                                   crm_campaign_name crm_country  \
0  TH_BKK_ctnoti_netcore_group_N_N_active_2024123...    Thailand   
1  EN_BKK_ctnoti_netcore_group_N_N_active_2024123...    Thailand   
2  EN_BKK_ctnoti_netcore_group_N_N_active_2024123...    Thailand   
3  TH

,activity_id,source_table,channel,crm_campaign_id,crm_campaign_name,crm_country,crm_restaurant_id,crm_restaurant_id_raw,crm_sent_datetime,crm_lang,...,crm_sent,crm_delivered,crm_total_clicked,crm_unique_clicked,crm_unique_conversions,crm_revenue,crm_ctr,crm_click_to_purchase,crm_delivered_to_purchase,crm_rev_per_click
0,CRM_8394b612f8e0,CRM_Data.xlsx,CRM,1350.0,TH_BKK_ctnoti_netcore_group_N_N_active_2024123...,Thailand,NaN,NaN,2024-12-31 11:13:00,TH,...,53947.0,45537.0,254.0,247.0,11.0,32800.0,0.0054,0.0445,0.000242,132.793522
1,CRM_652ec8390e85,CRM_Data.xlsx,CRM,1351.0,EN_BKK_ctnoti_netcore_group_N_N_active_2024123...,Thailand,NaN,NaN,2024-12-31 11:15:00,EN,...,42787.0,34220.0,302.0,284.0,5.0,15773.0,0.0083,0.0176,0.000146,55.538732
2,CRM_a09094249fb8,CRM_Data.xlsx,CRM,1353.0,EN_BKK_ctnoti_netcore_group_N_N_active_2024123...,Thailand,NaN,NaN,2024-12-31 14:05:00,EN,...,42819.0,6415.0,120.0,120.0,0.0,0.0,0.0187,0.0000,0.000000,0.000000
3,CRM_744905e26ba8,CRM_Data.xlsx,CRM,1352.0,TH_BKK_ctnoti_netcore_group_N_N_active_2024123...,Thailand,NaN,NaN,2024-12-31 14:05:00,TH,...,54066.0,12199.0,134.0,134.0,2.0,1221.0,0.0110,0.0149,0.000164,9.111940
4,CRM_8f8caca697a6,CRM_Data.xlsx,CRM,1355.0,EN_BKK_ctnoti_netcore_group_N_N_active_2024123...,Thailand,NaN,NaN,2024-12-31 16:00:00,EN,...,42866.0,34583.0,311.0,303.0,4.0,14525.0,0.0088,0.0132,0.000116,47.937294
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2349,CRM_549169cc64c9,CRM_Data.xlsx,CRM,4293.0,TH_BKK_ctnoti_netcore_group_N_N_active_2026011...,Thailand,NaN,NaN,2026-01-12 15:31:00,TH,...,77434.0,63910.0,209.0,205.0,4.0,6642.0,0.0032,0.0195,0.000063,32.400000
2350,CRM_394d842e1173,CRM_Data.xlsx,CRM,4294.0,EN_BKK_ctnoti_netcore_group_N_N_active_2026011...,Thailand,NaN,NaN,2026-01-12 15:34:00,EN,...,51469.0,42007.0,154.0,145.0,1.0,4464.0,0.0035,0.0069,0.000024,30.786207
2352,CRM_afdf3776ed4b,CRM_Data.xlsx,CRM,4296.0,EN_BKK_ctnoti_netcore_group_N_N_active_2026011...,Thailand,NaN,NaN,2026-01-12 17:15:00,EN,...,51378.0,42175.0,123.0,116.0,0.0,0.0,0.0028,0.0000,0.000000,0.000000
2353,CRM_4dd5aa0a6429,CRM_Data.xlsx,CRM,4297.0,TH_BKK_ctnoti_netcore_single_N_N_active_202601...,Thailand,NaN,NaN,2026-01-12 19:00:00,TH,...,77256.0,63907.0,178.0,170.0,3.0,5780.0,0.0027,0.0176,0.000047,34.000000


BUILD FACEBOOK ADS

In [4]:

def make_activity_id_fb(out: pd.DataFrame, source_name: str = "FB") -> pd.Series:
    base = (
        out["fb_ad_name"].fillna("").astype(str)
        + "|"
        + out["fb_start_date"].dt.strftime("%Y-%m-%d").fillna("")
        + "|"
        + out["fb_country"].fillna("").astype(str)
        + "|"
        + out["fb_restaurant_id"].fillna("").astype(str)
    )
    return base.apply(lambda s: f"{source_name}_" + hashlib.md5(s.encode("utf-8")).hexdigest()[:12])

def build_master_fb(fb_df: pd.DataFrame) -> pd.DataFrame:
    df = fb_df.copy()
    df.columns = df.columns.astype(str).str.replace("\ufeff", "", regex=False).str.strip()

    out = pd.DataFrame(index=df.index)

    # ── Meta ──────────────────────────────────────────────────────────────────
    out["source_table"]        = "fb_ads_valid.csv"
    out["channel"]             = "FB"

    # ── Ad identity ───────────────────────────────────────────────────────────
    out["fb_ad_name"]          = df.get("Ad name",          np.nan)
    out["fb_delivery_status"]  = df.get("Delivery status",  np.nan)
    out["fb_delivery_level"]   = df.get("Delivery level",   np.nan)
    out["fb_result_type"]      = df.get("Result type",      np.nan)
    out["fb_source"]           = df.get("Source",           np.nan)  # KOL / MP / etc.
    out["fb_type"]             = df.get("Type",             np.nan)  # IMG / VDO / etc.
    out["fb_campaign"]         = df.get("Campaign",         np.nan)
    out["fb_kol_id"]           = df.get("KOL_ID",          np.nan)
    out["fb_platform"]         = df.get("Platform",         np.nan)
    out["fb_content_date"]     = df.get("DATE",             np.nan)  # shoot / content date from ad name

    # ── Country ───────────────────────────────────────────────────────────────
    out["fb_country"]          = df.get("Country",          np.nan)  # already parsed upstream

    # ── Dates — column names corrected from old "Starts"/"Ends" ──────────────
    out["fb_start_date"]       = parse_date(df.get("Start_Date", pd.Series([np.nan] * len(df), index=df.index)))
    out["fb_end_date"]         = parse_date(df.get("End_Date",   pd.Series([np.nan] * len(df), index=df.index)))

    # Pre-computed date-derived fields — pass through, no recomputation needed
    out["fb_campaign_duration_days"] = to_num(df.get("Campaign Duration (Days)", pd.Series([np.nan] * len(df), index=df.index)))
    out["fb_year_month"]             = df.get("Year_Month", np.nan)  # e.g. "2025-06"

    # ── Restaurant IDs — corrected from old "Rest ID" ────────────────────────
    out["fb_restaurant_id"]    = to_num(df.get("Restaurant ID", pd.Series([np.nan] * len(df), index=df.index)))
    out["fb_restaurant_id_2"]  = to_num(df.get("Rest 2",        pd.Series([np.nan] * len(df), index=df.index)))
    out["fb_restaurant_id_3"]  = to_num(df.get("Rest 3",        pd.Series([np.nan] * len(df), index=df.index)))
    out["fb_restaurant_id_4"]  = to_num(df.get("Rest 4",        pd.Series([np.nan] * len(df), index=df.index)))
    out["fb_restaurant_id_5"]  = to_num(df.get("Rest 5",        pd.Series([np.nan] * len(df), index=df.index)))

    # Pre-parsed combined restaurant ID string from ad text (e.g. "1747+2102")
    # out["fb_restaurant_id_raw"] = df.get("Restaurant_ID_frm_adtext", np.nan)
    out["fb_restaurant_name"]   = df.get("Restaurant Name",           np.nan)

    # ── Metrics — raw counts ──────────────────────────────────────────────────
    out["fb_reach"]              = to_num(df.get("Reach",              pd.Series([np.nan] * len(df), index=df.index)))
    out["fb_impressions"]        = to_num(df.get("Impressions",        pd.Series([np.nan] * len(df), index=df.index)))
    out["fb_frequency"]          = to_num(df.get("Frequency",          pd.Series([np.nan] * len(df), index=df.index)))
    out["fb_results"]            = to_num(df.get("Results",            pd.Series([np.nan] * len(df), index=df.index)))
    out["fb_amount_spent_thb"]   = to_num(df.get("Amount spent (THB)", pd.Series([np.nan] * len(df), index=df.index)))
    out["fb_cost_per_result"]    = to_num(df.get("Cost per result",    pd.Series([np.nan] * len(df), index=df.index)))

    # ── Metrics — pre-computed, pass through as-is ────────────────────────────
    out["fb_ctr_proxy_pct"]      = to_num(df.get("CTR_Proxy_Pct",        pd.Series([np.nan] * len(df), index=df.index)))
    out["fb_cost_per_mille"]     = to_num(df.get("CPM",                  pd.Series([np.nan] * len(df), index=df.index)))
    out["fb_cost_per_reach"]     = to_num(df.get("Cost_per_Reach",       pd.Series([np.nan] * len(df), index=df.index)))
    out["fb_conversion_rate_pct"]= to_num(df.get("Conversion_Rate_Pct",  pd.Series([np.nan] * len(df), index=df.index)))

    # ── Activity ID ───────────────────────────────────────────────────────────
    out["activity_id"] = make_activity_id_fb(out, source_name="FB")

    # ── Final column order ────────────────────────────────────────────────────
    master_cols = [
        "activity_id",
        "source_table",
        "channel",
        # ad identity
        "fb_ad_name",
        "fb_delivery_status",
        "fb_delivery_level",
        "fb_result_type",
        "fb_format_type",
        "fb_source",
        "fb_type",
        "fb_campaign",
        "fb_kol_id",
        "fb_platform",
        "fb_content_date",
        # geography
        "fb_country",
        # dates
        "fb_start_date",
        "fb_end_date",
        "fb_campaign_duration_days",
        "fb_year_month",
        # restaurants
        "fb_restaurant_id",
        "fb_restaurant_id_2",
        "fb_restaurant_id_3",
        "fb_restaurant_id_4",
        "fb_restaurant_id_5",
        "fb_restaurant_id_raw",
        "fb_restaurant_name",
        # raw metrics
        "fb_reach",
        "fb_impressions",
        "fb_frequency",
        "fb_results",
        "fb_amount_spent_thb",
        "fb_cost_per_result",
        # pre-computed metrics
        "fb_ctr_proxy_pct",
        "fb_cost_per_mille",
        "fb_cost_per_reach",
        "fb_conversion_rate_pct",
    ]

    return out[[c for c in master_cols if c in out.columns]].copy()

# def build_master_fb(fb_df: pd.DataFrame) -> pd.DataFrame:
#     df = fb_df.copy()
#     df.columns = df.columns.astype(str).str.replace("\ufeff", "", regex=False).str.strip()
#     out = pd.DataFrame(index=df.index)

#     # -------- core --------
#     out["source_table"] = "fb_ads_valid.csv"   # change to your real file name
#     out["channel"] = "FB"

#     out["fb_ad_name"] = df.get("Ad name", np.nan)
#     out["fb_delivery_status"] = df.get("Delivery status", np.nan)
#     out["fb_delivery_level"] = df.get("Delivery level", np.nan)
#     out["fb_result_type"] = df.get("Result type", np.nan)

#     # Dates
#     out["fb_start_date"] = parse_date(df.get("Starts", pd.Series([np.nan] * len(df), index=df.index)))
#     out["fb_end_date"] = parse_date(df.get("Ends", pd.Series([np.nan] * len(df), index=df.index)))
    

#     # Restaurant keys (you already have Rest ID + Rest 2..5)
#     out["fb_restaurant_id"] = to_num(df.get("Rest ID", pd.Series([np.nan] * len(df), index=df.index)))

#     # Keep extra restaurant IDs as-is (no computation)
#     out["fb_restaurant_id_2"] = to_num(df.get("Rest 2", pd.Series([np.nan] * len(df), index=df.index)))
#     out["fb_restaurant_id_3"] = to_num(df.get("Rest 3", pd.Series([np.nan] * len(df), index=df.index)))
#     out["fb_restaurant_id_4"] = to_num(df.get("Rest 4", pd.Series([np.nan] * len(df), index=df.index)))
#     out["fb_restaurant_id_5"] = to_num(df.get("Rest 5", pd.Series([np.nan] * len(df), index=df.index)))

#     # Country: if you don't have a Country column, parse from Ad name if it’s encoded
#     # But user said: DO NOT compute new columns, so ONLY use existing Country col if present.
#     out["fb_country"] = df.get("Country", np.nan)

#     # -------- metrics (NO new derived cols) --------
#     out["fb_reach"] = to_num(df.get("Reach", pd.Series([np.nan] * len(df), index=df.index)))
#     out["fb_impressions"] = to_num(df.get("Impressions", pd.Series([np.nan] * len(df), index=df.index)))
#     out["fb_frequency"] = to_num(df.get("Frequency", pd.Series([np.nan] * len(df), index=df.index)))

#     out["fb_results"] = to_num(df.get("Results", pd.Series([np.nan] * len(df), index=df.index)))
#     out["fb_amount_spent_thb"] = to_num(df.get("Amount spent (THB)", pd.Series([np.nan] * len(df), index=df.index)))
#     out["fb_cost_per_result"] = to_num(df.get("Cost per result", pd.Series([np.nan] * len(df), index=df.index)))
#     out['fb_ctr_proxy_pct'] = to_num(df.get("CTR_Proxy_Pct", pd.Series([np.nan] * len(df), index=df.index)))
#     out['fb_cost_per_mille'] = to_num(df.get("CPM", pd.Series([np.nan] * len(df), index=df.index)))
#     out['fb_cost_per_reach'] = to_num(df.get("Cost_per_Reach", pd.Series([np.nan] * len(df), index=df.index)))

#     # -------- activity id --------
#     out["activity_id"] = make_activity_id_fb(out, source_name="FB")

#     # -------- final columns --------
#     master_cols = [
#         "activity_id", "source_table", "channel",
#         "fb_ad_name", "fb_delivery_status", "fb_delivery_level", "fb_result_type",
#         "fb_country",
#         "fb_start_date", "fb_end_date",
#         "fb_restaurant_id", "fb_restaurant_id_2", "fb_restaurant_id_3", "fb_restaurant_id_4", "fb_restaurant_id_5",
#         "fb_reach", "fb_impressions", "fb_frequency",
#         "fb_results", "fb_amount_spent_thb", "fb_cost_per_result", "fb_ctr_proxy_pct", "fb_cost_per_mille", "fb_cost_per_reach",
#     ]
#     return out[[c for c in master_cols if c in out.columns]].copy()


# --- usage ---
master_fb = build_master_fb(pd.read_csv(FB_PATH))

print("MASTER_FB shape:", master_fb.shape)
print(master_fb.head(5))
master_fb


MASTER_FB shape: (2438, 34)
       activity_id      source_table channel  \
0  FB_db27a1a9c78f  fb_ads_valid.csv      FB   
1  FB_f7c640cf02a7  fb_ads_valid.csv      FB   
2  FB_d82ac819134f  fb_ads_valid.csv      FB   
3  FB_3c358ca900ba  fb_ads_valid.csv      FB   
4  FB_f16097867288  fb_ads_valid.csv      FB   

                                          fb_ad_name fb_delivery_status  \
0  SRC=KOL|COUNTRY=TH|RID=1747+2102|RNAME=JUMBO-S...           inactive   
1  SRC=KOL|COUNTRY=TH|RID=1747+2102|RNAME=JUMBO-S...             active   
2  SRC=KOL|COUNTRY=TH|RID=1747+2102|RNAME=JUMBO-S...           inactive   
3  SRC=MP|COUNTRY=TH|RID=3181+2447+2252|RNAME=BEE...           inactive   
4  SRC=KOL|COUNTRY=TH|RID=5845+3905|RNAME=MAN_FU_...             active   

  fb_delivery_level     fb_result_type fb_source fb_type  fb_campaign  ...  \
0                ad  Website purchases       KOL     IMG          NaN  ...   
1                ad  Website purchases       KOL     IMG          NaN  ...  

,activity_id,source_table,channel,fb_ad_name,fb_delivery_status,fb_delivery_level,fb_result_type,fb_source,fb_type,fb_campaign,...,fb_reach,fb_impressions,fb_frequency,fb_results,fb_amount_spent_thb,fb_cost_per_result,fb_ctr_proxy_pct,fb_cost_per_mille,fb_cost_per_reach,fb_conversion_rate_pct
0,FB_db27a1a9c78f,fb_ads_valid.csv,FB,SRC=KOL|COUNTRY=TH|RID=1747+2102|RNAME=JUMBO-S...,inactive,ad,Website purchases,KOL,IMG,NaN,...,503,607,1.206759,1,22.78,22.780000,0.198807,37.528830,0.045288,0.164745
1,FB_f7c640cf02a7,fb_ads_valid.csv,FB,SRC=KOL|COUNTRY=TH|RID=1747+2102|RNAME=JUMBO-S...,active,ad,Website purchases,KOL,IMG,NaN,...,1481,2216,1.496286,4,115.17,28.792500,0.270088,51.972022,0.077765,0.180505
2,FB_d82ac819134f,fb_ads_valid.csv,FB,SRC=KOL|COUNTRY=TH|RID=1747+2102|RNAME=JUMBO-S...,inactive,ad,NaN,KOL,IMG,NaN,...,941,1207,1.282678,0,41.76,0.000000,0.000000,34.598177,0.044378,0.000000
3,FB_3c358ca900ba,fb_ads_valid.csv,FB,SRC=MP|COUNTRY=TH|RID=3181+2447+2252|RNAME=BEE...,inactive,ad,Website purchases,MP,IMG,MP - NOV'25,...,2831,4174,1.474391,10,146.85,14.685000,0.353232,35.182080,0.051872,0.239578
4,FB_f16097867288,fb_ads_valid.csv,FB,SRC=KOL|COUNTRY=TH|RID=5845+3905|RNAME=MAN_FU_...,active,ad,NaN,KOL,IMG,NaN,...,54,58,1.074074,0,2.40,0.000000,0.000000,41.379310,0.044444,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2433,FB_094628110d9b,fb_ads_valid.csv,FB,SRC=MP|COUNTRY=TH|RID=1622|RNAME=PANORAMA_CROW...,active,ad,Website purchases,MP,IMG,FESTIVE MP-DEC'25,...,1768,2334,1.320136,9,128.31,14.256667,0.509050,54.974293,0.072574,0.385604
2434,FB_18ef554f15d2,fb_ads_valid.csv,FB,SRC=HH|COUNTRY=TH|RID=3005|RNAME=GREAT_HARBOUR...,not_delivering,ad,NaN,HH,IMG,TOURIST-JAPAN,...,2553,3691,1.445750,0,455.92,0.000000,0.000000,123.522081,0.178582,0.000000
2435,FB_f13775165266,fb_ads_valid.csv,FB,SRC=HH|COUNTRY=TH|RID=4195|RNAME=JIM_THOMPSON_...,not_delivering,ad,NaN,HH,IMG,TOURIST-JAPAN,...,6712,7582,1.129619,0,498.98,0.000000,0.000000,65.811132,0.074341,0.000000
2436,FB_04de9ce11bf4,fb_ads_valid.csv,FB,SRC=HH|COUNTRY=TH|RID=4749|RNAME=SANEH_JAAN|TY...,archived,ad,NaN,HH,IMG,TOURIST-KOREA,...,1,1,1.000000,0,0.01,0.000000,0.000000,10.000000,0.010000,0.000000


In [5]:
master_db = pd.concat(
    [master_kol, master_crm, master_fb],
    ignore_index=True,
    sort=False
)

print(master_db.shape)
print(master_db.head())


(5984, 71)
        activity_id   source_table channel kol_platform    kol_id  \
0  KOL_78dd89a55819  KOL_Posts.csv     KOL    Instagram  HH-IN36Y   
1  KOL_d342d0c786bc  KOL_Posts.csv     KOL    Instagram  HH-INAAD   
2  KOL_3a536753bc0b  KOL_Posts.csv     KOL    Instagram  HH-INBEH   
3  KOL_efe32ed42d46  KOL_Posts.csv     KOL    Instagram  HH-INBEH   
4  KOL_46e9882811d4  KOL_Posts.csv     KOL    Instagram  HH-INBEH   

      kol_username kol_restaurant_id  \
0     365days2play              1893   
1      aayan_world              4503   
2  beatricechongyh               NaN   
3  beatricechongyh              4039   
4  beatricechongyh              5945   

                                kol_restaurant_name kol_country  \
0             Sirimahannop Asiatique The Riverfront     Bangkok   
1  Copper Beyond Buffet Gaysorn Amarin (Hungry Hub)     Bangkok   
2               Copper Beyond Buffet Gaysorn Amarin     Bangkok   
3                          Oishi Grand Siam Paragon     Bangkok  

In [6]:
import pandas as pd
import numpy as np

master = master_db.copy()

# -------------------------
# Step 1 — activity_start
# -------------------------
master["activity_start"] = pd.NaT

master.loc[master["channel"] == "CRM", "activity_start"] = pd.to_datetime(
    master.loc[master["channel"] == "CRM", "crm_sent_datetime"], errors="coerce"
)

master.loc[master["channel"] == "KOL", "activity_start"] = pd.to_datetime(
    master.loc[master["channel"] == "KOL", "kol_post_date"], errors="coerce"
)

master.loc[master["channel"] == "FB", "activity_start"] = pd.to_datetime(
    master.loc[master["channel"] == "FB", "fb_start_date"], errors="coerce"
)

# -------------------------
# Step 2 — activity_end
# -------------------------
master["activity_end"] = master["activity_start"]

# CRM: 48 hours
crm_mask = master["channel"] == "CRM"
master.loc[crm_mask, "activity_end"] = master.loc[crm_mask, "activity_start"] + pd.Timedelta(hours=48)

# KOL: 5 days
kol_mask = master["channel"] == "KOL"
master.loc[kol_mask, "activity_end"] = master.loc[kol_mask, "activity_start"] + pd.Timedelta(days=5)

# FB: cap at min(end_date, start+7d), else start+3d
fb_mask = master["channel"] == "FB"

fb_start = master.loc[fb_mask, "activity_start"]
fb_end = pd.to_datetime(master.loc[fb_mask, "fb_end_date"], errors="coerce")

fb_end_capped = fb_end.copy()
fb_end_capped = fb_end_capped.where(
    fb_end_capped.notna(),
    fb_start + pd.Timedelta(days=3)              # if missing end date
)
fb_end_capped = np.minimum(
    fb_end_capped,
    fb_start + pd.Timedelta(days=7)              # cap at 7 days maximum
)

master.loc[fb_mask, "activity_end"] = fb_end_capped

# Save
master.to_csv(OUT_DIR / "master_marketing_activties.csv", index=False)


In [7]:
master

,activity_id,source_table,channel,kol_platform,kol_id,kol_username,kol_restaurant_id,kol_restaurant_name,kol_country,kol_cuisine_type,...,fb_frequency,fb_results,fb_amount_spent_thb,fb_cost_per_result,fb_ctr_proxy_pct,fb_cost_per_mille,fb_cost_per_reach,fb_conversion_rate_pct,activity_start,activity_end
0,KOL_78dd89a55819,KOL_Posts.csv,KOL,Instagram,HH-IN36Y,365days2play,1893,Sirimahannop Asiatique The Riverfront,Bangkok,Thai-European,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-08-23,2024-08-28
1,KOL_d342d0c786bc,KOL_Posts.csv,KOL,Instagram,HH-INAAD,aayan_world,4503,Copper Beyond Buffet Gaysorn Amarin (Hungry Hub),Bangkok,International,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-09-05,2025-09-10
2,KOL_3a536753bc0b,KOL_Posts.csv,KOL,Instagram,HH-INBEH,beatricechongyh,NaN,Copper Beyond Buffet Gaysorn Amarin,Bangkok,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-08-18,2025-08-23
3,KOL_efe32ed42d46,KOL_Posts.csv,KOL,Instagram,HH-INBEH,beatricechongyh,4039,Oishi Grand Siam Paragon,Bangkok,Japanese,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-08-20,2025-08-25
4,KOL_46e9882811d4,KOL_Posts.csv,KOL,Instagram,HH-INBEH,beatricechongyh,5945,Kohaku Omakase,Bangkok,Japanese,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-08-23,2025-08-28
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5979,FB_094628110d9b,fb_ads_valid.csv,FB,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.320136,9.0,128.31,14.256667,0.50905,54.974293,0.072574,0.385604,2025-12-15,2025-12-22
5980,FB_18ef554f15d2,fb_ads_valid.csv,FB,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.445750,0.0,455.92,0.000000,0.00000,123.522081,0.178582,0.000000,2025-12-15,2025-12-18
5981,FB_f13775165266,fb_ads_valid.csv,FB,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.129619,0.0,498.98,0.000000,0.00000,65.811132,0.074341,0.000000,2025-12-15,2025-12-18
5982,FB_04de9ce11bf4,fb_ads_valid.csv,FB,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.000000,0.0,0.01,0.000000,0.00000,10.000000,0.010000,0.000000,2025-12-15,2025-12-18
